# BİM448 Veri Madenciliği ve Bilgi Keşfi - Proje Ödevi
## 2. Yöntem: BERT & BERTurk Tabanlı Duygu Analizi (9.000 İngilizce & 9.000 Türkçe Restoran/Kafe Yorumu)
- **Veri Kaynağı:** Google Drive (`/content/drive/MyDrive/BIM448_Duygu_Analizi_Projesi`)
- **İngilizce Model:** `gohbwj/sentiment-fine-tuned-yelp-3L` (Fine-tuned 3-Class BERT)
- **Türkçe Model:** `incidelen/bert-base-turkish-sentiment-analysis-cased` (`dbmdz/bert-base-turkish-cased` - BERTurk tabanlı 3-Class)

In [1]:
# 1. Gerekli kütüphanelerin kurulumu ve GPU kontrolü
!pip install -q transformers torch scikit-learn pandas

import os
import torch
import pandas as pd
from transformers import pipeline, logging
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

logging.set_verbosity_error()
pd.set_option('display.max_colwidth', None)

device = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else -1)
print("Aktif Hızlandırıcı Cihaz:", "CUDA GPU" if device == 0 else device)

Aktif Hızlandırıcı Cihaz: CUDA GPU


In [2]:
# 2. Google Drive Bağlantısı ve Veri Setlerinin Otomatik Çekilmesi
from google.colab import drive

# Google Drive'ı Colab ortamına bağla
drive.mount("/content/drive")

# Drive üzerindeki proje klasörü yolu
DRIVE_FOLDER = "/content/drive/MyDrive/BIM448_Duygu_Analizi_Projesi"

en_file_path = os.path.join(DRIVE_FOLDER, "english_restaurant_reviews_9000.csv")
tr_file_path = os.path.join(DRIVE_FOLDER, "turkish_restaurant_reviews_9000.csv")

# Veri setlerini Drive'dan oku
df_en = pd.read_csv(en_file_path)
df_tr = pd.read_csv(tr_file_path)

print(f"İngilizce Veri Seti Yüklendi: {df_en.shape} -> {en_file_path}")
print(f"Türkçe Veri Seti Yüklendi   : {df_tr.shape} -> {tr_file_path}")
display(df_en.head(2))
display(df_tr.head(2))

Mounted at /content/drive
İngilizce Veri Seti Yüklendi: (9000, 6) -> /content/drive/MyDrive/BIM448_Duygu_Analizi_Projesi/english_restaurant_reviews_9000.csv
Türkçe Veri Seti Yüklendi   : (9000, 6) -> /content/drive/MyDrive/BIM448_Duygu_Analizi_Projesi/turkish_restaurant_reviews_9000.csv


,id,place_name,place_category,rating,text,true_label
0,1,Taco Bell,"Tacos, Breakfast & Brunch, Restaurants, Fast Food, Mexican",5,"The wait wasn't long, the attendants were nice, the order was correct, AND the food very hot!! Excellent. Redeemed my recent experience at another Taco Bell. Thank",Positive
1,2,Oliver's,"American (Traditional), Vegan, Restaurants",5,Great restaurant! High quality food and beautiful surroundings. I wish we had more restaurants like this one in Santa Barbara,Positive


,id,place_name,place_category,rating,text,true_label
0,1,Kahve Dünyası - Büyükada,Kafe,1,Manzarası güzel diye oturmak istedik fakat hem personelin tavrı bakımından hem tuvaletlerin ve masaların temizliği açısından berbat. Uzun süredir bu kadar hijyenden uzak bir tuvalet görmemiştim. Kahve dünyası markasına hiç yakıştıramadım. Ek olarak kahve gurmesi değilim ama hayatımda içtiğim en kötü flat whitelardan biriydi.,Negative
1,2,Kilisli Fiko,Aile restoranı,1,Tamamen yorumlara ve puanina bakarak yolumu değiştirerek gittim. Vasat bir kebap yağ desen hiç yok meze diye sadece avuç içi kadar sumakli sogan domates koymuşlar. Bir de servisi ekmek arasi yapiyorlar. Kisacasi hiç memnun kalmadim.,Negative


In [3]:
# 3. Sınıf Bazlı Hata Analizi ve Değerlendirme Fonksiyonu
LABELS = ["Positive", "Neutral", "Negative"]

def evaluate_bert(df, pred_col, title):
    y_true = df["true_label"]
    y_pred = df[pred_col]
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)

    print("=" * 65)
    print(f" {title}")
    print("=" * 65)
    print(f"Genel Doğruluk (Accuracy)   : %{acc * 100:.2f} ({(y_true == y_pred).sum():,} / {len(df):,} doğru)")
    print(f"Toplam Hatalı Tahmin Sayısı : {(y_true != y_pred).sum():,} / {len(df):,}\n")

    print("Sınıf Bazlı Hatalı Tahmin Analizi (Gerçek Sınıfa Göre):")
    for i, lbl in enumerate(LABELS):
        tot = int(cm[i].sum())
        cor = int(cm[i, i])
        err = tot - cor
        mis = ", ".join([f"{LABELS[j]}'e kayan: {int(cm[i, j])}" for j in range(len(LABELS)) if j != i])
        print(f"  - [{lbl:8s}] Toplam: {tot} | Doğru: {cor:4d} | Hatalı (Yanlış): {err:4d} (%{err/tot*100:5.2f} hata) -> ({mis})")

    print("\nKarmaşıklık Matrisi (Confusion Matrix):")
    cm_df = pd.DataFrame(cm, index=[f"Gerçek {l}" for l in LABELS], columns=[f"Tahmin {l}" for l in LABELS])
    display(cm_df)
    print("\nDetaylı Sınıflandırma Raporu:")
    print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

In [4]:
%%time
# 4. İNGİLİZCE BERT ANALİZİ (9.000 Yorum)
EN_MODEL = "gohbwj/sentiment-fine-tuned-yelp-3L"
EN_MAP = {"LABEL_0": "Negative", "LABEL_1": "Positive", "LABEL_2": "Neutral"}

pipe_en = pipeline(
    "sentiment-analysis",
    model=EN_MODEL,
    device=device,
    truncation=True,
    max_length=512
)

out_en = pipe_en(list(df_en["text"].astype(str)), batch_size=64)

df_en["bert_raw_label"] = [o["label"] for o in out_en]
df_en["bert_confidence"] = [round(float(o["score"]), 4) for o in out_en]
df_en["bert_pred"] = [EN_MAP[o["label"]] for o in out_en]

# Sonuçları Google Drive klasörüne kaydet
en_out_path = os.path.join(DRIVE_FOLDER, "bert_english_results.csv")
df_en.to_csv(en_out_path, index=False, encoding="utf-8")
print(f"İngilizce BERT sonuçları Drive'a kaydedildi: {en_out_path}\n")

evaluate_bert(df_en, "bert_pred", "İNGİLİZCE VERİ SETİ - BERT SONUÇLARI (9.000 Yorum)")

config.json:   0%|          | 0.00/769 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  268MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

... (3 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

İngilizce BERT sonuçları Drive'a kaydedildi: /content/drive/MyDrive/BIM448_Duygu_Analizi_Projesi/bert_english_results.csv

 İNGİLİZCE VERİ SETİ - BERT SONUÇLARI (9.000 Yorum)
Genel Doğruluk (Accuracy)   : %80.96 (7,286 / 9,000 doğru)
Toplam Hatalı Tahmin Sayısı : 1,714 / 9,000

Sınıf Bazlı Hatalı Tahmin Analizi (Gerçek Sınıfa Göre):
  - [Positive] Toplam: 3000 | Doğru: 2613 | Hatalı (Yanlış):  387 (%12.90 hata) -> (Neutral'e kayan: 371, Negative'e kayan: 16)
  - [Neutral ] Toplam: 3000 | Doğru: 2288 | Hatalı (Yanlış):  712 (%23.73 hata) -> (Positive'e kayan: 320, Negative'e kayan: 392)
  - [Negative] Toplam: 3000 | Doğru: 2385 | Hatalı (Yanlış):  615 (%20.50 hata) -> (Positive'e kayan: 13, Neutral'e kayan: 602)

Karmaşıklık Matrisi (Confusion Matrix):


,Tahmin Positive,Tahmin Neutral,Tahmin Negative
Gerçek Positive,2613,371,16
Gerçek Neutral,320,2288,392
Gerçek Negative,13,602,2385



Detaylı Sınıflandırma Raporu:
              precision    recall  f1-score   support

    Positive     0.8870    0.8710    0.8789      3000
     Neutral     0.7016    0.7627    0.7309      3000
    Negative     0.8539    0.7950    0.8234      3000

    accuracy                         0.8096      9000
   macro avg     0.8142    0.8096    0.8111      9000
weighted avg     0.8142    0.8096    0.8111      9000

CPU times: user 17.3 s, sys: 1.51 s, total: 18.8 s
Wall time: 34 s


In [5]:
%%time
# 5. TÜRKÇE BERTurk ANALİZİ (9.000 Yorum)
TR_MODEL = "incidelen/bert-base-turkish-sentiment-analysis-cased"
TR_MAP = {"Negative": "Negative", "Neutral": "Neutral", "Positive": "Positive"}

pipe_tr = pipeline(
    "sentiment-analysis",
    model=TR_MODEL,
    device=device,
    truncation=True,
    max_length=512
)

out_tr = pipe_tr(list(df_tr["text"].astype(str)), batch_size=64)

df_tr["bert_raw_label"] = [o["label"] for o in out_tr]
df_tr["bert_confidence"] = [round(float(o["score"]), 4) for o in out_tr]
df_tr["bert_pred"] = [TR_MAP[o["label"]] for o in out_tr]

# Sonuçları Google Drive klasörüne kaydet
tr_out_path = os.path.join(DRIVE_FOLDER, "bert_turkish_results.csv")
df_tr.to_csv(tr_out_path, index=False, encoding="utf-8")
print(f"Türkçe BERTurk sonuçları Drive'a kaydedildi: {tr_out_path}\n")

evaluate_bert(df_tr, "bert_pred", "TÜRKÇE VERİ SETİ - BERTurk SONUÇLARI (9.000 Yorum)")

config.json:   0%|          | 0.00/862 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Türkçe BERTurk sonuçları Drive'a kaydedildi: /content/drive/MyDrive/BIM448_Duygu_Analizi_Projesi/bert_turkish_results.csv

 TÜRKÇE VERİ SETİ - BERTurk SONUÇLARI (9.000 Yorum)
Genel Doğruluk (Accuracy)   : %72.11 (6,490 / 9,000 doğru)
Toplam Hatalı Tahmin Sayısı : 2,510 / 9,000

Sınıf Bazlı Hatalı Tahmin Analizi (Gerçek Sınıfa Göre):
  - [Positive] Toplam: 3000 | Doğru: 2366 | Hatalı (Yanlış):  634 (%21.13 hata) -> (Neutral'e kayan: 478, Negative'e kayan: 156)
  - [Neutral ] Toplam: 3000 | Doğru: 1353 | Hatalı (Yanlış): 1647 (%54.90 hata) -> (Positive'e kayan: 180, Negative'e kayan: 1467)
  - [Negative] Toplam: 3000 | Doğru: 2771 | Hatalı (Yanlış):  229 (% 7.63 hata) -> (Positive'e kayan: 47, Neutral'e kayan: 182)

Karmaşıklık Matrisi (Confusion Matrix):


,Tahmin Positive,Tahmin Neutral,Tahmin Negative
Gerçek Positive,2366,478,156
Gerçek Neutral,180,1353,1467
Gerçek Negative,47,182,2771



Detaylı Sınıflandırma Raporu:
              precision    recall  f1-score   support

    Positive     0.9125    0.7887    0.8461      3000
     Neutral     0.6721    0.4510    0.5398      3000
    Negative     0.6306    0.9237    0.7495      3000

    accuracy                         0.7211      9000
   macro avg     0.7384    0.7211    0.7118      9000
weighted avg     0.7384    0.7211    0.7118      9000

CPU times: user 25.8 s, sys: 1.14 s, total: 26.9 s
Wall time: 32.4 s
